In [ ]:
from pathlib import Path
import shutil
import torch
import pandas as pd
from tqdm import tqdm

import fiftyone as fo

import fiftyone.zoo as foz
from fiftyone import ViewField as F

from detect_DINO import (
    load_grounding_dino_model,
    detect_jaguars_in_dataset,
    filter_high_confidence_detections,
    get_samples_with_detections,
    create_dataset_from_directory,
    select_best_detection,
)

from segment_SAM2 import (
    load_sam2_model,
    segment_detections,
    segment_positive_detections,
    export_segmented_dataset,
)

# Upload the Dataset
This script creates a FiftyOne dataset from the local screenshots directory. It scans for all image files in the screenshots folder and its subdirectories.

In [ ]:
# Set up paths
image_dir = Path('../../data/intermediate/v1/screenshots')

# Create a new dataset
dataset_name = "jaguar_detection"
if fo.dataset_exists(dataset_name):
    print(f"Loading existing dataset: {dataset_name}")
    dataset = fo.load_dataset(dataset_name)
    print(f"Dataset contains {len(dataset)} samples")
else:
    print(f"Creating new dataset: {dataset_name}")
    dataset = fo.Dataset(name=dataset_name, persistent=True)
    
    # Add all images from directory and subdirectories
    image_paths = []
    for ext in ["*.jpg", "*.jpeg", "*.png", "*.JPG", "*.JPEG", "*.PNG"]:
        image_paths.extend(image_dir.rglob(ext))
    
    print(f"Found {len(image_paths)} images")
    
    samples = []
    for img_path in tqdm(image_paths, desc="Adding images"):
        sample = fo.Sample(filepath=str(img_path))
        samples.append(sample)
    
    dataset.add_samples(samples)
    print(f"Added {len(dataset)} samples to dataset")

# Run Grounding-Dino

In [ ]:
DETECTION_TYPE = "body"
USE_TEST_DATASET = False
DEVICE = "cpu"
TEXT_PROMPT = "jaguar"

print(f"Using {DEVICE} device")

working_dataset = dataset
print(f"Working with {'test' if USE_TEST_DATASET else 'full'} dataset ({len(working_dataset)} samples)")

model = load_grounding_dino_model(device=DEVICE, text_prompt=TEXT_PROMPT)
raw_bboxes_name = f"raw_bboxes_{DETECTION_TYPE}"

detect_jaguars_in_dataset(
    working_dataset,
    model=model,
    label_field=raw_bboxes_name,
    confidence_threshold=0.2,
)

print(f"✓ Model inference complete on {len(working_dataset)} samples")

In [ ]:
select_best_detection(working_dataset, detection_type=DETECTION_TYPE, raw_bboxes_field_name=raw_bboxes_name)
print(f"✓ Best detection selection complete")

In [ ]:
detection_field = "bboxes_body" if DETECTION_TYPE == "body" else "bboxes_head"
samples_with_detections = working_dataset.exists(detection_field)
total_samples = len(working_dataset)
detected_samples = len(samples_with_detections)

print(f"Detection Results:")
print(f"  Total samples: {total_samples}")
print(f"  Samples with detections: {detected_samples}")
print(f"  Samples without detections: {total_samples - detected_samples}")
print(f"  Detection rate: {detected_samples/total_samples*100:.1f}%")

In [ ]:
import shutil

# Save positive and negative detections into separate subfolders with folder-based prefixes
detection_field = "bboxes_body" if DETECTION_TYPE == "body" else "bboxes_head"

positive_dir = Path('../../data/intermediate/v1/screenshots/positive_detections')
negative_dir = Path('../../data/intermediate/v1/screenshots/negative_detections')

positive_dir.mkdir(parents=True, exist_ok=True)
negative_dir.mkdir(parents=True, exist_ok=True)

positive_count = 0
negative_count = 0

for sample in tqdm(working_dataset, desc="Organizing detections"):
    img_path = Path(sample.filepath)
    
    # Get the parent folder name to use as prefix
    folder_name = img_path.parent.name
    prefixed_filename = f"{folder_name}_{img_path.name}"
    
    # Check if sample has detections
    if sample[detection_field] and sample[detection_field].detections:
        # Positive detection
        dest = positive_dir / prefixed_filename
        positive_count += 1
    else:
        # Negative detection
        dest = negative_dir / prefixed_filename
        negative_count += 1
    
    # Copy file to appropriate folder
    shutil.copy(img_path, dest)

print(f"✓ Detections organized into folders")
print(f"  Total processed: {len(working_dataset)}")
print(f"  Positive: {positive_count} images -> {positive_dir}")
print(f"  Negative: {negative_count} images -> {negative_dir}")
print(f"  Positive folder: {len(list(positive_dir.glob('*')))} files")
print(f"  Negative folder: {len(list(negative_dir.glob('*')))} files")

## Segment with SAM2

In [ ]:
positive_dataset = segment_positive_detections(working_dataset, detection_type=DETECTION_TYPE, device=DEVICE)

In [ ]:
storage_dir = Path('../../data/intermediate/v1/fo_jaguars')

post_segmentation_view = fo.load_dataset("jaguar_detection")
positive_field = "bboxes_body"
positive_samples_view = post_segmentation_view.exists(positive_field)

export_segmented_dataset(
    positive_samples_view,
    export_dir=storage_dir / "segmented_jaguars",
    rel_dir=image_dir,
)

session = fo.launch_app(positive_samples_view)

# Add Jaguar ID Labels
Match sample filenames to jaguar IDs from the cleaned labels CSV and add them as metadata to each sample.


## Step 1: Load Labels and Create Mapping
Load the cleaned labels CSV and create a mapping from filename to jaguar ID for verification.


In [ ]:
import pandas as pd

labels_path = Path('../../data/intermediate/v1/cleaned_labels.csv')
print(f"Loading labels from {labels_path}...")
labels_df = pd.read_csv(labels_path)

# Convert FILE PATH to mapping keys: "sites/SITE 11/CAM B/DSCF0016.AVI" -> "SITE_11_CAM_B_DSCF0016"
def extract_filepath_key(file_path):
    file_path = str(file_path)
    # Remove "sites/" prefix
    if "sites/" in file_path:
        file_path = file_path.split("sites/", 1)[1]
    # Remove extension
    file_path = str(Path(file_path).with_suffix(''))
    # Replace path separators and spaces with underscores
    key = file_path.replace("/", "_").replace(" ", "_")
    return key

# Create mapping using extracted filepaths
file_to_jaguar = {}
for idx, row in labels_df.iterrows():
    filepath_key = extract_filepath_key(row['FILE PATH'])
    jaguar_id = row['JAGUAR ID']
    file_to_jaguar[filepath_key] = jaguar_id

print(f"Loaded {len(labels_df)} labels from CSV")
print(f"Found {len(file_to_jaguar)} file-to-jaguar mappings")

print("\nSample file-to-jaguar mappings:")
for i, (fpath, jid) in enumerate(list(file_to_jaguar.items())[:10]):
    print(f"  {fpath} -> {jid}")


## Step 2: Load Dataset and Check Filename Matches
Load the positive samples dataset and verify that filenames can be matched to jaguar IDs.


In [ ]:
segmented_jaguars = fo.load_dataset()

## Step 3: Apply Labels and Visualize
Add the jaguar ID labels to all matched samples and launch FiftyOne for visualization.


## Export Labeled Dataset
Load the segmented jaguars dataset, add jaguar ID labels, and export as a new labeled dataset.


In [ ]:
# Load the segmented jaguars dataset
storage_dir = Path('../../data/intermediate/v1/fo_jaguars')
labeled_dataset = fo.Dataset.from_dir(
    dataset_dir=str(storage_dir / "segmented_jaguars"),
    dataset_type=fo.types.FiftyOneDataset,
    name="segmented_jaguars_with_labels"
)

print(f"Loaded {len(labeled_dataset)} segmented jaguar samples")

# Add jaguar ID labels to all samples
labeled_count = 0
unlabeled_count = 0

for sample in tqdm(labeled_dataset, desc="Adding jaguar ID labels"):
    # Extract filepath and convert to underscore format
    filepath = str(Path(sample.filepath).with_suffix(''))
    filepath_key = filepath.replace("/", "_").replace(" ", "_")
    
    # Look up the jaguar ID
    if filepath_key in file_to_jaguar:
        jaguar_id = file_to_jaguar[filepath_key]
        sample["jaguar_id"] = jaguar_id
        sample.save()
        labeled_count += 1
    else:
        unlabeled_count += 1

print(f"\n✓ Labels added:")
print(f"  Labeled: {labeled_count} samples")
print(f"  Unlabeled: {unlabeled_count} samples")
print(f"  Coverage: {labeled_count/(labeled_count+unlabeled_count)*100:.1f}%")

# Export the labeled dataset with media
export_dir = Path('../../data/intermediate/v1/fo_jaguars/labeled_segmented_jaguars')
export_dir.parent.mkdir(parents=True, exist_ok=True)

labeled_dataset.export(
    export_dir=str(export_dir),
    dataset_type=fo.types.FiftyOneDataset,
    export_media=True
)

print(f"\n✓ Labeled dataset exported with media files:")
print(f"  Location: {export_dir}")
print(f"  Total samples: {len(labeled_dataset)}")
print(f"  With segmentations and jaguar ID labels")

# Launch FiftyOne app to visualize the labeled dataset
session = fo.launch_app(labeled_dataset)


In [ ]:
# Check package versions
import fiftyone
import transformers
print(f"fiftyone: {fiftyone.__version__}")
print(f"transformers: {transformers.__version__}")